# Experiments Analysis

Retrieve and filter LangSmith evaluation runs for flexible post-hoc analysis.

In [1]:
import sys
import os
import json
from collections import Counter

sys.path.insert(0, os.path.abspath(".."))

from typing import Any, Callable, Dict, List, Optional

from langsmith import Client

client = Client()

In [2]:
def load_experiment_runs(
    experiment_prefix: str = "",
    dataset_name: str = "Dataset v1.",
) -> List[Dict[str, Any]]:
    """
    Retrieve all evaluation runs from LangSmith experiments and return them
    as a flat list of dicts ready for filtering and inspection.

    Each record contains:
        experiment         — experiment (project) name
        task_id            — the eval task identifier
        thread_id          — the pipeline checkpoint thread id
        run_id             — LangSmith run id
        scores             — dict of {score_key: float} from all evaluators
        <score_key>        — every score also flattened to top level for easy filtering
        final_code         — generated code from the pipeline state
        retrieval_context  — RAG context from the pipeline state
        comments           — dict of {feedback_key: [parsed_data | string]} from evaluators
                             Contains: code_statements_score, rag_statements_score, etc.
        outputs            — raw LangSmith run outputs (full pre_computed_state inside)

    Args:
        experiment_prefix: Only include experiments whose name starts with this string.
                           Empty string means include all experiments for the dataset.
        dataset_name:      LangSmith dataset name used to scope experiments.
    """
    records: List[Dict[str, Any]] = []

    projects = list(client.list_projects(reference_dataset_name=dataset_name))
    if experiment_prefix:
        projects = [p for p in projects if p.name.startswith(experiment_prefix)]

    print(f"Found {len(projects)} experiment(s) matching prefix '{experiment_prefix}'")

    for project in projects:
        runs = list(client.list_runs(project_name=project.name, is_root=True))

        if not runs:
            continue

        # Bulk-fetch all feedback for this project in one call
        run_ids = [str(r.id) for r in runs]
        feedback_by_run: Dict[str, Dict[str, Any]] = {rid: {} for rid in run_ids}
        for fb in client.list_feedback(run_ids=run_ids):
            rid = str(fb.run_id)

            if fb.score is not None:
                feedback_by_run[rid][fb.key] = fb.score
            
            # Store the comment for each feedback key as well
            if fb.comment:
                if f"{fb.key}_comment" not in feedback_by_run[rid]:
                    feedback_by_run[rid][f"{fb.key}_comment"] = fb.comment

        for run in runs:
            rid = str(run.id)
            feedback_data = feedback_by_run.get(rid, {})
            scores = {k: v for k, v in feedback_data.items() if not k.endswith("_comment")}
            outputs = run.outputs or {}
            pre_state = outputs.get("pre_computed_state") or {}

            task_id = (
                str(outputs.get("task_id") or "").strip()
                or str(pre_state.get("task_id") or "").strip()
            )

            # Extract and parse statements from feedback comments
            # All statement data is stored in comments with keys like:
            # code_statements_score_comment, rag_statements_score_comment, etc.
            comments_dict = {}
            
            for key, value in feedback_data.items():
                if key.endswith("_comment"):
                    comment_key = key.replace("_comment", "")
                    
                    # Try to parse as JSON; if it fails, keep raw comment string
                    if value:
                        try:
                            parsed = json.loads(value)
                            comments_dict[comment_key] = parsed
                        except (json.JSONDecodeError, ValueError):
                            # If JSON parsing fails, keep raw comment string
                            comments_dict[comment_key] = value
                    else:
                        comments_dict[comment_key] = value

            record: Dict[str, Any] = {
                "experiment": project.name,
                "task_id": task_id,
                "thread_id": outputs.get("thread_id", ""),
                "run_id": rid,
                "scores": scores,
                "final_code": pre_state.get("final_code") or pre_state.get("generated_code") or "",
                "retrieval_context": pre_state.get("retrieval_context", ""),
                "comments": comments_dict,
                "outputs": outputs,
            }
            # Flatten scores to top level for convenient filtering
            record.update(scores)

            records.append(record)

    print(f"Loaded {len(records)} total run(s)")
    return records

In [3]:
def filter_runs(
    records: List[Dict[str, Any]],
    **conditions: Any,
) -> List[Dict[str, Any]]:
    """
    Filter records by field values.

    Each keyword argument is a field name. The value can be:
    - A scalar     → exact equality match  (e.g., code_execution_score=0.0)
    - A callable   → predicate on the field (e.g., task_id=lambda v: "add" in v)

    Missing fields evaluate to None against the condition.

    Example:
        filter_runs(records, code_statements_score=1.0, code_execution_score=0.0)
        filter_runs(records, task_id=lambda v: v.startswith("add"))
    """
    result = []
    for record in records:
        if all(
            (cond(record.get(key)) if callable(cond) else record.get(key) == cond)
            for key, cond in conditions.items()
        ):
            result.append(record)
    return result


def display_runs(
    records: List[Dict[str, Any]],
    show_code: bool = False,
    show_context: bool = False,
    code_preview_chars: int = 600,
    context_preview_chars: int = 400,
) -> None:
    """
    Pretty-print a list of run records.

    Args:
        records:              Output of load_experiment_runs / filter_runs.
        show_code:            Print a preview of final_code.
        show_context:         Print a preview of retrieval_context.
        code_preview_chars:   Max chars to show for code preview.
        context_preview_chars: Max chars to show for context preview.
    """
    SEP = "─" * 70
    print(f"{len(records)} record(s)\n{SEP}")

    for i, r in enumerate(records, 1):
        scores = r.get("scores", {})
        score_str = "  ".join(
            f"{k}={v:.2f}" for k, v in sorted(scores.items()) if v is not None
        )
        print(f"\n[{i}]  task_id={r.get('task_id', '?')!r}")
        print(f"     experiment: {r.get('experiment', '?')}")
        print(f"     thread_id:  {r.get('thread_id', '?')}")
        print(f"     scores:     {score_str or '(none)'}")

        if show_code:
            code = r.get("final_code", "") or ""
            snippet = code[:code_preview_chars]
            ellipsis = "..." if len(code) > code_preview_chars else ""
            print(f"     final_code:\n{snippet}{ellipsis}")

        if show_context:
            ctx = r.get("retrieval_context", "") or ""
            snippet = ctx[:context_preview_chars]
            ellipsis = "..." if len(ctx) > context_preview_chars else ""
            print(f"     retrieval_context:\n{snippet}{ellipsis}")

    print(f"\n{SEP}")

## Usage

In [4]:
# --- Load ---
# Empty prefix loads all experiments for the dataset.
# Supply a prefix string to scope to a specific experiment family.
EXPERIMENT_PREFIX = "COMPLEX CONTEXT PLAN PROMPT: v1"       # e.g. "2) With concision"
DATASET_NAME = "Dataset v2."

records = load_experiment_runs(
    experiment_prefix=EXPERIMENT_PREFIX,
    dataset_name=DATASET_NAME,
)

Found 1 experiment(s) matching prefix 'COMPLEX CONTEXT PLAN PROMPT: v1'
Loaded 5 total run(s)


In [5]:

# --- Verify all scores and statements are in comments ---
print("=== Verifying all data is contained in comments ===\n")

r = records[0]
print(f"Task: {r['task_id']}\n")
print(f"Record keys: {list(r.keys())}\n")

print("Comments structure:")
for comment_type, comment_value in r['comments'].items():
    print(f"\n[{comment_type}]")
    print(f"  Type: {type(comment_value).__name__}")
    
    if isinstance(comment_value, dict):
        print(f"  Keys: {list(comment_value.keys())}")
        if 'pass' in comment_value:
            print(f"  Pass: {comment_value['pass']}")
    elif isinstance(comment_value, list):
        print(f"  List length: {len(comment_value)}")
        if comment_value and isinstance(comment_value[0], dict):
            print(f"  First item keys: {list(comment_value[0].keys())}")
            if "status" in comment_value[0]:
                print(f"  Sample status: {comment_value[0]['status']}")
    else:
        print(f"  Raw value: {str(comment_value)[:100]}")

# Count how many comments are parsed vs raw
parsed_count = sum(1 for r in records for c in r['comments'].values() if not isinstance(c, str))
raw_count = sum(1 for r in records for c in r['comments'].values() if isinstance(c, str))
total_comments = sum(len(r['comments']) for r in records)

print(f"\n\nSummary across all {len(records)} records:")
print(f"  Total comment fields: {total_comments}")
print(f"  Parsed (JSON/Lists): {parsed_count}")
print(f"  Raw (string): {raw_count}")

# Verify key statement types exist
statement_types = set()
for r in records:
    for key in r.get('comments', {}).keys():
        if 'statement' in key:
            statement_types.add(key)

print(f"\nStatement types found: {statement_types}")

=== Verifying all data is contained in comments ===

Task: archive_stale_tasks

Record keys: ['experiment', 'task_id', 'thread_id', 'run_id', 'scores', 'final_code', 'retrieval_context', 'comments', 'outputs', 'code_execution_score', 'code_statements_score', 'rag_statements_score']

Comments structure:

[code_execution_score]
  Type: dict
  Keys: ['pass', 'output', 'errors']
  Pass: True

[code_statements_score]
  Type: list
  List length: 6
  First item keys: ['statement', 'status', 'evidence', 'reasoning']
  Sample status: Present

[rag_statements_score]
  Type: list
  List length: 6
  First item keys: ['statement', 'status', 'evidence', 'reasoning']
  Sample status: Not present


Summary across all 5 records:
  Total comment fields: 15
  Parsed (JSON/Lists): 15
  Raw (string): 0

Statement types found: {'code_statements_score', 'rag_statements_score'}


In [6]:
def analyze_available_statement_data(records):
    """
    Analyze what statement data is available across all records in comments.
    Helps us understand coverage and identify missing data patterns.
    """
    from collections import defaultdict
    
    # Track statement data availability
    records_with_statements = defaultdict(int)
    records_without_statements = 0
    comment_keys_count = defaultdict(int)
    
    for record in records:
        comments = record.get('comments', {}) or {}
        if not comments:
            records_without_statements += 1
            continue
        
        has_any_statements = False
        for key in comments.keys():
            comment_keys_count[key] += 1
            if 'statement' in key:
                has_any_statements = True
                records_with_statements[key] += 1
        
        if not has_any_statements:
            records_without_statements += 1
    
    print("\n" + "="*90)
    print("DATA AVAILABILITY ANALYSIS")
    print("="*90)
    
    print(f"\nTotal records: {len(records)}")
    print(f"Records with statement data: {len(records) - records_without_statements}")
    print(f"Records without statements: {records_without_statements}")
    
    print(f"\n📊 Comment fields across all records:")
    for key in sorted(comment_keys_count.keys()):
        count = comment_keys_count[key]
        pct = (count / len(records) * 100) if records else 0
        is_statement = '✓' if 'statement' in key else '✗'
        print(f"  {is_statement} {key:40} {count:6} records ({pct:5.1f}%)")
    
    print(f"\n📋 Statement data availability:")
    for key in sorted(records_with_statements.keys()):
        count = records_with_statements[key]
        pct = (count / len(records) * 100) if records else 0
        print(f"  • {key:40} {count:6} records ({pct:5.1f}%)")

analyze_available_statement_data(records)



DATA AVAILABILITY ANALYSIS

Total records: 5
Records with statement data: 5
Records without statements: 0

📊 Comment fields across all records:
  ✗ code_execution_score                          5 records (100.0%)
  ✓ code_statements_score                         5 records (100.0%)
  ✓ rag_statements_score                          5 records (100.0%)

📋 Statement data availability:
  • code_statements_score                         5 records (100.0%)
  • rag_statements_score                          5 records (100.0%)


In [7]:
# Count the number of CODE statements grouped by task_id
print("\n=== Counting CODE statements by task_id ===\n")

task_statement_counts = Counter()

for r in records:
    task_id = r.get("task_id", "unknown")
    comments = r.get("comments", {}) or {}
    statements = comments.get("code_statements_score")

    # Statements should come from comments, not from top-level property
    if isinstance(statements, list):
        task_statement_counts[task_id] += len(statements)

print("Statements count by task_id:")
for task_id, count in task_statement_counts.most_common():
    print(f"  {task_id!r}: {count} statement(s)")


=== Counting CODE statements by task_id ===

Statements count by task_id:
  'propagate_status_done': 8 statement(s)
  'vague_emergency_escalation': 7 statement(s)
  'archive_stale_tasks': 6 statement(s)
  'create_task_with_blocks_and_relation': 6 statement(s)
  'update_imminent_tasks': 6 statement(s)


## Code & RAG Statement Failure Analysis

Analyze CODE and RAG statement failures with proper status/reasoning structure.


In [8]:
def extract_statement_failures(statements_list):
    """
    Parse failures from statement list with proper status/reasoning structure.
    
    Statement format:
      {
        "statement": str,        # the claim being checked
        "status": str,           # "Present" | "Wrong" | "Not present"
        "evidence": str,         # verbatim quote from context
        "reasoning": str         # explanation of the status
      }
    
    A statement FAILS when status is "Wrong" or "Not present".
    Returns dict mapping failure_key -> {count, status_type, reasoning, examples}
    """
    failures = {}
    
    if not isinstance(statements_list, list):
        return failures
    
    for stmt in statements_list:
        if not isinstance(stmt, dict):
            continue
        
        status = stmt.get("status", "").strip()
        if status not in ("Wrong", "Not present"):
            continue  # Only track failures
        
        statement_text = stmt.get("statement", "Unknown statement")[:100]
        reasoning = stmt.get("reasoning", "No explanation provided")
        evidence = stmt.get("evidence", "NONE")
        
        # Composite key: status + reasoning
        failure_key = f"{status}: {reasoning}"
        
        if failure_key not in failures:
            failures[failure_key] = {
                "count": 0,
                "status_type": status,
                "reasoning": reasoning,
                "examples": []
            }
        
        failures[failure_key]["count"] += 1
        if len(failures[failure_key]["examples"]) < 2:
            failures[failure_key]["examples"].append({
                "statement": statement_text,
                "evidence": evidence[:200]
            })
    
    return failures


print("✅ Statement failure extractor loaded.")


✅ Statement failure extractor loaded.


In [9]:
def _group_by_task(records):
    """Reusable: Group records by task_id."""
    from collections import defaultdict
    records_by_task = defaultdict(list)
    for record in records:
        task_id = record.get("task_id", "unknown")
        records_by_task[task_id].append(record)
    return records_by_task


def _analyze_statement_failures_generic(records, statement_key, auxiliary_field=None):
    """
    Generic statement failure analyzer for CODE or RAG.
    
    Args:
        records: List of run records
        statement_key: e.g. "code_statements_score" or "rag_statements_score"
        auxiliary_field: e.g. "final_code" or "retrieval_context" for case samples
    
    Returns:
        (failures_by_task, coverage_by_task, total_by_task, case_samples)
    """
    from collections import defaultdict
    
    failures_by_task = defaultdict(dict)
    coverage_by_task = defaultdict(int)
    total_by_task = defaultdict(int)
    case_samples = defaultdict(lambda: defaultdict(list))
    
    records_by_task = _group_by_task(records)
    
    for task_id, task_records in records_by_task.items():
        total_by_task[task_id] = len(task_records)
        
        for record in task_records:
            comments = record.get("comments", {}) or {}
            stmts = comments.get(statement_key)
            run_id = record.get("run_id", "")
            
            if not isinstance(stmts, list):
                continue
            
            coverage_by_task[task_id] += 1
            record_failures = extract_statement_failures(stmts)
            
            for failure_key, failure_info in record_failures.items():
                if failure_key not in failures_by_task[task_id]:
                    failures_by_task[task_id][failure_key] = {
                        "count": 0,
                        "status_type": failure_info["status_type"],
                        "reasoning": failure_info["reasoning"]
                    }
                
                failures_by_task[task_id][failure_key]["count"] += failure_info["count"]
                
                sample = {
                    "run_id": run_id,
                    "examples": failure_info["examples"],
                }
                if auxiliary_field:
                    sample["data"] = record.get(auxiliary_field, "")[:300]
                
                case_samples[task_id][failure_key].append(sample)
    
    return failures_by_task, coverage_by_task, total_by_task, case_samples


def _display_statement_failures_generic(failures_by_task, coverage_by_task, total_by_task, 
                                        case_samples, title="STATEMENT FAILURES"):
    """
    Generic display function for statement failures.
    """
    print("\n" + "="*90)
    print(f"{title} - BY TASK (WITH COVERAGE)")
    print("="*90)

    for task_id in sorted(failures_by_task.keys()):
        failures = failures_by_task[task_id]
        coverage = coverage_by_task.get(task_id, 0)
        total = total_by_task.get(task_id, 0)
        total_failures = sum(f["count"] for f in failures.values())
        
        print(f"\n📋 TASK: {task_id}")
        print(f"   Records: {coverage}/{total} have statements ({coverage/total*100:.1f}%)")
        print(f"   Total failures: {total_failures} | Failure types: {len(failures)}")
        print(f"   {'-'*82}")
        
        if not failures:
            print(f"   ✓ No failures found in {coverage} records")
            continue
        
        for failure_key in sorted(failures.keys(), key=lambda k: failures[k]["count"], reverse=True):
            info = failures[failure_key]
            pct = (info["count"] / total_failures * 100) if total_failures > 0 else 0
            
            print(f"\n   [{info['status_type']}] {info['reasoning']}")
            print(f"        Count: {info['count']} ({pct:.1f}%)")
            
            samples = case_samples[task_id][failure_key]
            if samples and samples[0]["examples"]:
                ex = samples[0]["examples"][0]
                print(f"        Example statement: {ex['statement']}")
                print(f"        Evidence: {ex['evidence']}")

    print(f"\n{'='*90}\n")


print("✅ Modular utilities loaded.")

✅ Modular utilities loaded.


### CODE Statement Failure Analysis

Analyze CODE statement failures by task and failure type.

In [10]:
def analyze_code_failures(records):
    """Analyze CODE statement failures by task and failure type."""
    return _analyze_statement_failures_generic(records, "code_statements_score", "final_code")


code_failures_by_task, code_coverage, code_total, code_case_samples = analyze_code_failures(records)
_display_statement_failures_generic(code_failures_by_task, code_coverage, code_total, 
                                    code_case_samples, "CODE STATEMENT FAILURES")



CODE STATEMENT FAILURES - BY TASK (WITH COVERAGE)

📋 TASK: archive_stale_tasks
   Records: 1/1 have statements (100.0%)
   Total failures: 3 | Failure types: 3
   ----------------------------------------------------------------------------------

   [Wrong] Uses 'Last Reviewed' and 'before' instead of 'Created' and 'on_or_before'.
        Count: 1 (33.3%)
        Example statement: The stale filter uses POST https://api.notion.com/v1/databases/{database_id}/query with {'property':
        Evidence: "property": "Last Reviewed", "date": {"before": thirty_days_ago}

   [Wrong] Updates status property instead of setting the 'archived' flag.
        Count: 1 (33.3%)
        Example statement: For every matched page id, PATCH https://api.notion.com/v1/pages/{page_id} is called with top-level 
        Evidence: "properties": {"Status": {"select": {"name": "Done"}}}

   [Wrong] Payload includes an extra 'type' key inside the parent object.
        Count: 1 (33.3%)
        Example statement: C

In [ ]:
def _filter_and_group_by_task(records, filter_fn):
    """
    Generic: Filter records by condition and group by task.
    Returns dict: {task_id: [filtered_records]}
    """
    by_task = {}
    for record in records:
        if filter_fn(record):
            task_id = record.get("task_id", "unknown")
            if task_id not in by_task:
                by_task[task_id] = []
            by_task[task_id].append(record)
    return by_task


def _display_by_task_results(by_task, title, format_record_fn):
    """
    Generic: Display results grouped by task with custom formatting per record.
    format_record_fn: callable(record) -> dict with keys for display
    """
    if not by_task:
        print(f"\n✅ GOOD NEWS: No cases for {title}!")
        return
    
    if not callable(format_record_fn):
        raise TypeError(f"format_record_fn must be callable, got {type(format_record_fn).__name__}")
    
    print(f"\n{'='*100}")
    print(title)
    print(f"{'='*100}")
    print(f"\nTotal cases: {sum(len(v) for v in by_task.values())} across {len(by_task)} task(s)\n")
    
    for task_id in sorted(by_task.keys()):
        task_results = by_task[task_id]
        print(f"📋 TASK: {task_id}")
        print(f"   {len(task_results)} case(s)")
        print(f"   {'-'*96}")
        
        for i, r in enumerate(task_results[:3], 1):
            formatted = format_record_fn(r)
            if not isinstance(formatted, dict):
                raise TypeError(f"format_record_fn must return dict, got {type(formatted).__name__}")
            print(f"\n   [{i}]")
            for key, val in formatted.items():
                if val and not key.startswith("_"):  # Skip display keys starting with _
                    print(f"      {key}: {val}")
        
        if len(task_results) > 3:
            print(f"   ... and {len(task_results) - 3} more cases\n")
    
    print(f"{'='*100}\n")

In [20]:
def _format_code_execution_case(record):
    """Format a record for code-executed display."""
    comments = record.get("comments", {}) or {}
    code_stmts = comments.get("code_statements_score")
    stmt_failures = extract_statement_failures(code_stmts) if isinstance(code_stmts, list) else {}
    
    return {
        "Run": record.get("run_id", ""),
        "Experiment": record.get("experiment", ""),
        "Failures": sum(f["count"] for f in stmt_failures.values()),
        "Code Preview": record.get("final_code", "")[:200],
    }


code_executed_by_task = _filter_and_group_by_task(
    records,
    lambda r: (r.get("scores", {}).get("code_execution_error", r.get("scores", {}).get("execution_error")) == 0.0
               and (lambda c=r.get("comments", {}).get("code_statements_score"): bool(extract_statement_failures(c) if isinstance(c, list) else False))())
)
_display_by_task_results(code_executed_by_task, "CODE EXECUTED BUT STATEMENTS NOT PERFECT", _format_code_execution_case)


✅ GOOD NEWS: No cases for CODE EXECUTED BUT STATEMENTS NOT PERFECT!


In [21]:
def _format_failed_perfect_case(record):
    """Format a record for failed-but-perfect-statements display."""
    return {
        "Run": record.get("run_id", ""),
        "Experiment": record.get("experiment", ""),
        "Statements Checked": len(record.get("comments", {}).get("code_statements_score", [])),
        "Code Preview": record.get("final_code", "")[:200],
    }


failed_perfect_by_task = _filter_and_group_by_task(
    records,
    lambda r: (r.get("scores", {}).get("code_execution_error", r.get("scores", {}).get("execution_error")) == 1.0
               and not extract_statement_failures(r.get("comments", {}).get("code_statements_score", []))
               and isinstance(r.get("comments", {}).get("code_statements_score"), list))
)
_display_by_task_results(failed_perfect_by_task, "CODE DID NOT EXECUTE BUT STATEMENTS ALL PERFECT", _format_failed_perfect_case)


✅ GOOD NEWS: No cases for CODE DID NOT EXECUTE BUT STATEMENTS ALL PERFECT!


### RAG Statement Failure Analysis

Analyze RAG statement failures by task and failure type.

In [15]:
def analyze_rag_failures(records):
    """Analyze RAG statement failures by task and failure type."""
    return _analyze_statement_failures_generic(records, "rag_statements_score", "retrieval_context")


rag_failures_by_task, rag_coverage, rag_total, rag_case_samples = analyze_rag_failures(records)
_display_statement_failures_generic(rag_failures_by_task, rag_coverage, rag_total, 
                                    rag_case_samples, "RAG STATEMENT FAILURES")


RAG STATEMENT FAILURES - BY TASK (WITH COVERAGE)

📋 TASK: archive_stale_tasks
   Records: 1/1 have statements (100.0%)
   Total failures: 6 | Failure types: 6
   ----------------------------------------------------------------------------------

   [Not present] The provided context contains no information regarding Python code or date arithmetic logic.
        Count: 1 (16.7%)
        Example statement: Python datetime arithmetic computes a threshold date as current date minus timedelta(days=30).
        Evidence: NONE

   [Not present] The context does not define a query filter for 'Created' or 'on_or_before' logic.
        Count: 1 (16.7%)
        Example statement: The stale filter uses POST https://api.notion.com/v1/databases/{database_id}/query with {'property':
        Evidence: NONE

   [Wrong] The context explicitly states 'in_trash' replaces the deprecated 'archived' property.
        Count: 1 (16.7%)
        Example statement: For every matched page id, PATCH https://api.no

### Combined CODE vs RAG Analysis

Compare CODE vs RAG failures side-by-side for ALL tasks.

In [23]:
def comprehensive_statement_analysis(records):
    """
    Compare CODE vs RAG failures side-by-side for ALL tasks.
    Shows data coverage, failures, and overlap for all records.
    """
    from collections import Counter
    
    analysis = {}
    
    # Use existing grouping utility to avoid duplication
    records_by_task = _group_by_task(records)
    
    # Analyze each task
    for task_id, task_records in records_by_task.items():
        analysis[task_id] = {
            "total_runs": len(task_records),
            "code_data_coverage": 0,  # records with code statements
            "rag_data_coverage": 0,   # records with rag statements
            "code_failures": 0,
            "rag_failures": 0,
            "both_failed": 0,
            "code_only": 0,
            "rag_only": 0,
            "neither_failed": 0,
            "code_failure_reasons": Counter(),
            "rag_failure_reasons": Counter(),
        }
        
        for record in task_records:
            comments = record.get("comments", {}) or {}
            
            # Extract code statements
            code_stmts = comments.get("code_statements_score")
            code_has_data = isinstance(code_stmts, list)
            code_has_failures = False
            
            if code_has_data:
                analysis[task_id]["code_data_coverage"] += 1
                code_failures = extract_statement_failures(code_stmts)
                if code_failures:
                    code_has_failures = True
                    analysis[task_id]["code_failures"] += 1
                    for key, info in code_failures.items():
                        analysis[task_id]["code_failure_reasons"][f"[{info['status_type']}] {info['reasoning']}"] += info["count"]
            
            # Extract RAG statements
            rag_stmts = comments.get("rag_statements_score")
            rag_has_data = isinstance(rag_stmts, list)
            rag_has_failures = False
            
            if rag_has_data:
                analysis[task_id]["rag_data_coverage"] += 1
                rag_failures = extract_statement_failures(rag_stmts)
                if rag_failures:
                    rag_has_failures = True
                    analysis[task_id]["rag_failures"] += 1
                    for key, info in rag_failures.items():
                        analysis[task_id]["rag_failure_reasons"][f"[{info['status_type']}] {info['reasoning']}"] += info["count"]
            
            # Update overlap counters (only for records with both types of data)
            if code_has_data and rag_has_data:
                if code_has_failures and rag_has_failures:
                    analysis[task_id]["both_failed"] += 1
                elif code_has_failures:
                    analysis[task_id]["code_only"] += 1
                elif rag_has_failures:
                    analysis[task_id]["rag_only"] += 1
                else:
                    analysis[task_id]["neither_failed"] += 1
    
    return analysis


task_analysis = comprehensive_statement_analysis(records)

print("\n" + "="*100)
print("COMPREHENSIVE FAILURE ANALYSIS - ALL TASKS WITH DATA COVERAGE")
print("="*100)

for task_id in sorted(task_analysis.keys()):
    info = task_analysis[task_id]
    total = info["total_runs"]
    code_coverage = info["code_data_coverage"]
    rag_coverage = info["rag_data_coverage"]
    code_pct = (info["code_failures"] / code_coverage * 100) if code_coverage > 0 else 0
    rag_pct = (info["rag_failures"] / rag_coverage * 100) if rag_coverage > 0 else 0
    
    print(f"\n{'═'*100}")
    print(f"📌 TASK: {task_id}")
    print(f"{'═'*100}")
    print(f"  Total runs: {total}")
    print(f"\n  DATA COVERAGE:")
    print(f"    Code statements:   {code_coverage:3}/{total} records ({code_coverage/total*100:5.1f}%)")
    print(f"    RAG statements:    {rag_coverage:3}/{total} records ({rag_coverage/total*100:5.1f}%)")
    
    print(f"\n  FAILURES (among records with data):")
    print(f"    Code failures:     {info['code_failures']:3}/{code_coverage} ({code_pct:5.1f}%)")
    print(f"    RAG failures:      {info['rag_failures']:3}/{rag_coverage} ({rag_pct:5.1f}%)")
    
    # Overlap only for records with both types of data
    records_with_both = info["both_failed"] + info["code_only"] + info["rag_only"] + info["neither_failed"]
    if records_with_both > 0:
        print(f"\n  FAILURE OVERLAP (records with both code AND rag data = {records_with_both}):")
        print(f"    • Both failed:     {info['both_failed']:3}")
        print(f"    • Code only:       {info['code_only']:3}")
        print(f"    • RAG only:        {info['rag_only']:3}")
        print(f"    • Neither failed:  {info['neither_failed']:3}")
    
    # Top CODE failures
    if info["code_failure_reasons"]:
        print(f"\n  TOP CODE FAILURE REASONS:")
        for reason, count in info["code_failure_reasons"].most_common(3):
            pct = (count / info["code_failures"] * 100) if info["code_failures"] > 0 else 0
            print(f"    • {reason}")
            print(f"      {count} occurrences ({pct:.1f}% of code failures)")
    else:
        print(f"\n  Code failures: NONE")
    
    # Top RAG failures
    if info["rag_failure_reasons"]:
        print(f"\n  TOP RAG FAILURE REASONS:")
        for reason, count in info["rag_failure_reasons"].most_common(3):
            pct = (count / info["rag_failures"] * 100) if info["rag_failures"] > 0 else 0
            print(f"    • {reason}")
            print(f"      {count} occurrences ({pct:.1f}% of rag failures)")
    else:
        print(f"\n  RAG failures: NONE")

print(f"\n{'═'*100}\n")


COMPREHENSIVE FAILURE ANALYSIS - ALL TASKS WITH DATA COVERAGE

════════════════════════════════════════════════════════════════════════════════════════════════════
📌 TASK: archive_stale_tasks
════════════════════════════════════════════════════════════════════════════════════════════════════
  Total runs: 1

  DATA COVERAGE:
    Code statements:     1/1 records (100.0%)
    RAG statements:      1/1 records (100.0%)

  FAILURES (among records with data):
    Code failures:       1/1 (100.0%)
    RAG failures:        1/1 (100.0%)

  FAILURE OVERLAP (records with both code AND rag data = 1):
    • Both failed:       1
    • Code only:         0
    • RAG only:          0
    • Neither failed:    0

  TOP CODE FAILURE REASONS:
    • [Wrong] Uses 'Last Reviewed' and 'before' instead of 'Created' and 'on_or_before'.
      1 occurrences (100.0% of code failures)
    • [Wrong] Updates status property instead of setting the 'archived' flag.
      1 occurrences (100.0% of code failures)
    • [

### Statements

In [17]:
def inspect_statement_failures(records, task_id, statement_type="code", max_samples=10):
    """
    Drill down into specific statement failures for detailed inspection.
    
    Args:
        records: List of run records
        task_id: Filter by this task
        statement_type: "code" or "rag"
        max_samples: Max failure cases to show
    
    Returns structured failure cases with full details.
    """
    from collections import defaultdict
    
    score_key = f"{statement_type}_statements_score"
    failure_cases = defaultdict(list)
    
    for record in records:
        if record.get("task_id") != task_id:
            continue
        
        comments = record.get("comments", {}) or {}
        stmts = comments.get(score_key)
        if not isinstance(stmts, list):
            continue
        
        stmt_failures = extract_statement_failures(stmts)
        if not stmt_failures:
            continue
        
        for failure_key, failure_info in stmt_failures.items():
            for example in failure_info["examples"]:
                failure_cases[failure_key].append({
                    "run_id": record.get("run_id", ""),
                    "thread_id": record.get("thread_id", ""),
                    "statement": example["statement"],
                    "evidence": example["evidence"],
                    "code": record.get("final_code", "") if statement_type == "code" else None,
                    "context": record.get("retrieval_context", "") if statement_type == "rag" else None,
                })
    
    return failure_cases


def show_statement_failures(records, task_id, statement_type="code"):
    """
    Pretty-print detailed statement failure cases.
    """
    cases = inspect_statement_failures(records, task_id, statement_type)
    
    print(f"\n{'='*100}")
    print(f"DETAILED {statement_type.upper()} STATEMENT FAILURES: Task={task_id}")
    print(f"{'='*100}\n")
    
    for failure_key in sorted(cases.keys()):
        case_list = cases[failure_key]
        print(f"{'─'*100}")
        print(f"❌ {failure_key}")
        print(f"   Total occurrences: {len(case_list)}")
        print(f"{'─'*100}\n")
        
        # Show first 3 detailed cases
        for i, case in enumerate(case_list[:3], 1):
            print(f"   Case {i}: Run {case['run_id']}")
            print(f"   ├─ Statement: {case['statement']}")
            print(f"   ├─ Evidence: {case['evidence']}")
            
            if statement_type == "code" and case["code"]:
                print(f"   └─ Code (first 500 chars):\n       {case['code'][:500]}")
            elif statement_type == "rag" and case["context"]:
                print(f"   └─ Context (first 500 chars):\n       {case['context'][:500]}")
            print()

print("\n✅ Failure inspection tools ready!")
print("\nUsage examples:")
print("  show_statement_failures(records, task_id='add_task', statement_type='code')")
print("  show_statement_failures(records, task_id='add_task', statement_type='rag')")



✅ Failure inspection tools ready!

Usage examples:
  show_statement_failures(records, task_id='add_task', statement_type='code')
  show_statement_failures(records, task_id='add_task', statement_type='rag')


### Plans

In [19]:
from collections import defaultdict

# Group and display plans by task_id (look in multiple common locations)
plans_by_task = defaultdict(list)

for record in records:
    task_id = record.get("task_id", "unknown")
    outputs = record.get("outputs", {}) or {}
    pre_state = outputs.get("pre_computed_state", {}) or {}

    # Look for plan in several places where it may be stored
    candidates = {
        "outputs.plan": outputs.get("plan"),
        "outputs.request_plan": outputs.get("request_plan"),
        "pre_computed_state.plan": pre_state.get("plan"),
        "pre_computed_state.request_plan": pre_state.get("request_plan"),
    }

    # Pick the first non-empty string-like plan
    plan_source = None
    plan = None
    for src, val in candidates.items():
        if isinstance(val, str) and val.strip():
            plan_source = src
            plan = val.strip()
            break
        # If plan may be structured (dict), stringify it
        if isinstance(val, (dict, list)) and val:
            plan_source = src
            plan = json.dumps(val, indent=2) if isinstance(val, (dict, list)) else str(val)
            break

    if plan:
        plans_by_task[task_id].append({
            "run_id": record.get("run_id", ""),
            "experiment": record.get("experiment", ""),
            "plan_source": plan_source,
            "plan": plan
        })

print("\n" + "="*100)
print("PLANS RETRIEVED FROM RECORDS - GROUPED BY TASK")
print("="*100)

if not plans_by_task:
    print("No plans found in any record outputs or pre_computed_state.\n")
else:
    for task_id in sorted(plans_by_task.keys()):
        plans = plans_by_task[task_id]
        print(f"\n📋 TASK: {task_id}")
        print(f"   Total plans: {len(plans)}")
        print(f"   {'-'*96}")
        
        for i, plan_entry in enumerate(plans, 1):
            snippet = plan_entry['plan']
            ellipsis = "..." if len(snippet) < len(plan_entry['plan']) else ""
            print(f"\n   [{i}] Experiment: {plan_entry['experiment']}")
            print(f"       Run ID: {plan_entry['run_id']}")
            print(f"       Source: {plan_entry['plan_source']}")
            print(f"       Plan (preview):\n{snippet}{ellipsis}\n")

print("="*100 + "\n")


PLANS RETRIEVED FROM RECORDS - GROUPED BY TASK

📋 TASK: archive_stale_tasks
   Total plans: 1
   ------------------------------------------------------------------------------------------------

   [1] Experiment: COMPLEX CONTEXT PLAN PROMPT: v1-ef87150c
       Run ID: 019d029d-5caa-7023-a6af-18772420ba1d
       Source: pre_computed_state.request_plan
       Plan (preview):
## Execution Plan: Archive Stale Tasks

Here's a sequential execution plan to find tasks older than 30 days, archive them, and add a comment.

**Step 1: Query for Tasks Older Than 30 Days**

1.  **INPUTS:** Database ID (assumed user-provided). Current date.
2.  **ENDPOINT:** `POST /v1/databases/query`
3.  **SCHEMA:**
    ```json
    {
      "filter": {
        "property": "Last Reviewed",
        "date": {
          "before": "YYYY-MM-DD" // Calculate date 30 days ago
        }
      }
    }
    ```
4.  **VALIDATION:**
    *   The input must be a valid Database ID.
    *   The "Last Reviewed" property must exist in

### All in one

In [ ]:
import re
import json

def _find_plan(record):
    outputs = record.get("outputs", {}) or {}
    pre_state = outputs.get("pre_computed_state", {}) or {}

    candidates = {
        "outputs.plan": outputs.get("plan"),
        "outputs.request_plan": outputs.get("request_plan"),
        "pre_computed_state.plan": pre_state.get("plan"),
        "pre_computed_state.request_plan": pre_state.get("request_plan"),
    }

    for src, val in candidates.items():
        if isinstance(val, str) and val.strip():
            return src, val.strip()
        if isinstance(val, (dict, list)) and val:
            return src, json.dumps(val, indent=2)
    return None, None

def _find_queries(text):
    if not text:
        return []
    urls = set(re.findall(r"https?://[^\s'\"`]+", text))
    # also capture common API path patterns like /v1/databases/{...}/query
    paths = set(re.findall(r"/v1/[A-Za-z0-9_/{}-]+", text))
    results = sorted(urls | paths)
    return results

SEP = "\n" + ("=" * 100) + "\n"

for i, record in enumerate(records, 1):
    run_id = record.get("run_id", "")
    task_id = record.get("task_id", "")
    experiment = record.get("experiment", "")
    code = record.get("final_code", "") or ""
    context = record.get("retrieval_context", "") or ""
    src, plan = _find_plan(record)

    # search for queries/urls in code, context, and plan
    found = set()
    for txt in (code, context, plan or ""):
        found.update(_find_queries(txt))

    print(SEP)
    print(f"[{i}] run_id={run_id}  task_id={task_id}  experiment={experiment}\n")
    print(">>> CODE:")
    print(code or "(none)")
    #print("\n>>> CONTEXT:")
    #print(context or "(none)")
    print("\n>>> PLAN SOURCE:")
    print(src or "(none)")
    print("\n>>> PLAN:")
    print(plan or "(none)")
    print("\n>>> QUERIES / URLS FOUND:")
    if found:
        for q in sorted(found):
            print(f"  - {q}")
    else:
        print("  (none found)")
    print("\n>>> RAW OUTPUTS (pre_computed_state preview):")
    outputs = record.get("outputs", {}) or {}
    pre_state = outputs.get("pre_computed_state", {}) or {}
    # pretty-print a small preview of pre_computed_state
    try:
        preview = json.dumps(pre_state, indent=2) if pre_state else "{}"
    except Exception:
        preview = str(pre_state)
    print(preview)
print(SEP)



[1] run_id=019d029d-5caa-7023-a6af-18772420ba1d  task_id=archive_stale_tasks  experiment=COMPLEX CONTEXT PLAN PROMPT: v1-ef87150c

>>> CODE:
import os
import dotenv
import requests
import sys
from datetime import datetime, timedelta

dotenv.load_dotenv()

def archive_stale_tasks(token: str = os.getenv('NOTION_TOKEN'), db_id: str = os.getenv('NOTION_TASKS_DATABASE_ID')) -> None:
    """
    Finds tasks in the specified database where 'Last Reviewed' is older than 30 days,
    updates their status to 'Done', and adds a comment.
    """
    if not token or not db_id:
        raise ValueError("Missing NOTION_TOKEN or NOTION_TASKS_DATABASE_ID environment variables.")

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Notion-Version": "2022-06-28"
    }

    # Calculate date 30 days ago
    thirty_days_ago = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')

    # Query for stale tasks
    query_url = f"https://api.not

In [24]:
print("\n" + "="*100)
print("STATEMENT FAILURE ANALYSIS SUMMARY")
print("="*100)

# Summary stats from analyze_* functions
code_tasks_with_failures = sum(1 for t, f in code_failures_by_task.items() if f)
rag_tasks_with_failures = sum(1 for t, f in rag_failures_by_task.items() if f)
total_code_failures = sum(sum(f[k]["count"] for k in f) for f in code_failures_by_task.values())
total_rag_failures = sum(sum(f[k]["count"] for k in f) for f in rag_failures_by_task.values())

print(f"\n📊 OVERVIEW:")
print(f"   Total records analyzed: {len(records)}")
print(f"   Total tasks: {len(set(r.get('task_id') for r in records))}")
print(f"\n   CODE STATEMENTS:")
# Handle both dict and int coverage values safely
if isinstance(code_coverage, dict):
    code_tasks_with_data = sum(1 for c in code_coverage.values() if c > 0)
else:
    code_tasks_with_data = int(code_coverage)

# rag_coverage was overwritten as int in this session; rebuild a task map
if isinstance(rag_coverage, int):
    rag_coverage = {task: 1 for task in rag_total.keys()}

print(f"      Tasks with statement data: {code_tasks_with_data}")
print(f"      Tasks with failures: {code_tasks_with_failures}")
print(f"      Total failures: {total_code_failures}")
print(f"\n   RAG STATEMENTS:")
print(f"      Tasks with statement data: {len([t for t, c in rag_coverage.items() if c > 0])}")
print(f"      Tasks with failures: {rag_tasks_with_failures}")
print(f"      Total failures: {total_rag_failures}")

print(f"\n📋 ANALYSIS FUNCTIONS:")
print(f"   1. analyze_code_failures(records)")
print(f"      → CODE failures by task + data coverage")
print(f"      → Returns: (failures_by_task, coverage_by_task, total_by_task, case_samples)")
print(f"\n   2. analyze_rag_failures(records)")
print(f"      → RAG failures by task + data coverage")
print(f"      → Returns: (failures_by_task, coverage_by_task, total_by_task, case_samples)")
print(f"\n   3. comprehensive_statement_analysis(records)")
print(f"      → Compare CODE vs RAG side-by-side with overlap analysis")
print(f"      → Returns: dict with coverage, failures, and overlap stats")
print(f"\n   4. show_statement_failures(records, task_id, statement_type)")
print(f"      → Drill into specific failures with full evidence")
print(f"      → statement_type: 'code' or 'rag'")

print(f"\n🔍 TASK COMPARISON:")
print(f"\n   Task Name                       | Total | Code Data | RAG Data | Code Fail | RAG Fail")
print(f"   {'-'*95}")
for task_id in sorted(task_analysis.keys()):
    total = task_analysis[task_id]["total_runs"]
    code_cov = task_analysis[task_id]["code_data_coverage"]
    rag_cov = task_analysis[task_id]["rag_data_coverage"]
    code_fail = task_analysis[task_id]["code_failures"]
    rag_fail = task_analysis[task_id]["rag_failures"]
    print(f"   {task_id:31} | {total:5} | {code_cov:9} | {rag_cov:8} | {code_fail:9} | {rag_fail:8}")

print(f"\n" + "="*100 + "\n")



STATEMENT FAILURE ANALYSIS SUMMARY

📊 OVERVIEW:
   Total records analyzed: 5
   Total tasks: 5

   CODE STATEMENTS:
      Tasks with statement data: 1
      Tasks with failures: 4
      Total failures: 14

   RAG STATEMENTS:
      Tasks with statement data: 5
      Tasks with failures: 5
      Total failures: 30

📋 ANALYSIS FUNCTIONS:
   1. analyze_code_failures(records)
      → CODE failures by task + data coverage
      → Returns: (failures_by_task, coverage_by_task, total_by_task, case_samples)

   2. analyze_rag_failures(records)
      → RAG failures by task + data coverage
      → Returns: (failures_by_task, coverage_by_task, total_by_task, case_samples)

   3. comprehensive_statement_analysis(records)
      → Compare CODE vs RAG side-by-side with overlap analysis
      → Returns: dict with coverage, failures, and overlap stats

   4. show_statement_failures(records, task_id, statement_type)
      → Drill into specific failures with full evidence
      → statement_type: 'code' or